# Extract PECsw Data from xAquaticRisk HDF5 Store

This notebook extracts PECsw (Predicted Environmental Concentration in Surface Water) data from the xAquaticRisk model output and exports it to Excel.

## 1. Import Required Libraries

In [ ]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

print("Libraries imported successfully")

## 2. Define File Paths

In [ ]:
# Input HDF5 file
hdf5_path = Path(r"C:\LocalWork\xAquaticRisk\run\Test_Run_aqRisk\mcs\X34JQLGOQ6HWJ5H6CA\store\arr.dat")

# Output Excel file
output_path = Path(r"C:\LocalWork\xAquaticRisk\run\Test_Run_aqRisk\mcs\X34JQLGOQ6HWJ5H6CA\PECsw_output.xlsx")

print(f"Input file exists: {hdf5_path.exists()}")
print(f"Input file: {hdf5_path}")
print(f"Output file: {output_path}")

## 3. Explore HDF5 File Structure

In [ ]:
def explore_hdf5(file_path, max_depth=10):
    """Recursively explore and print HDF5 file structure."""
    def print_structure(name, obj):
        depth = name.count("/")
        if depth > max_depth:
            return
        indent = "  " * depth
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}📊 Dataset: {name}")
            print(f"{indent}   Shape: {obj.shape}, Dtype: {obj.dtype}")
            if obj.attrs:
                print(f"{indent}   Attributes: {dict(obj.attrs)}")
        elif isinstance(obj, h5py.Group):
            print(f"{indent}📁 Group: {name}")
    
    with h5py.File(file_path, 'r') as f:
        print("="*80)
        print("HDF5 FILE STRUCTURE")
        print("="*80)
        f.visititems(print_structure)
        print("="*80)

explore_hdf5(hdf5_path)

## 4. Find PECsw-Related Datasets

In [ ]:
def find_pecsw_datasets(file_path):
    """Find all datasets that might contain PECsw data."""
    pecsw_datasets = []
    
    def search(name, obj):
        if isinstance(obj, h5py.Dataset):
            name_lower = name.lower()
            keywords = ['pec', 'conc', 'exposure', 'toxswa', 'concentration']
            if any(keyword in name_lower for keyword in keywords):
                pecsw_datasets.append({
                    'path': name,
                    'shape': obj.shape,
                    'dtype': obj.dtype
                })
    
    with h5py.File(file_path, 'r') as f:
        f.visititems(search)
    
    return pecsw_datasets

pecsw_datasets = find_pecsw_datasets(hdf5_path)
print(f"Found {len(pecsw_datasets)} potential PECsw dataset(s):\n")
for i, ds in enumerate(pecsw_datasets, 1):
    print(f"{i}. Path: {ds['path']}")
    print(f"   Shape: {ds['shape']}")
    print(f"   Dtype: {ds['dtype']}\n")

## 5. Extract PECsw Data

In [ ]:
def extract_pecsw_data(file_path, dataset_name=None):
    """Extract PECsw data from the HDF5 file."""
    with h5py.File(file_path, 'r') as f:
        if dataset_name is None:
            possible_paths = [
                'CascadeToxswa/PECsw',
                'CascadeToxswa/ConcentrationSW',
                'CascadeToxswa/Concentration',
                'PECsw',
                'Concentration'
            ]
            
            for path in possible_paths:
                if path in f:
                    dataset_name = path
                    print(f"✓ Found dataset at: {path}")
                    break
            
            if dataset_name is None:
                datasets = find_pecsw_datasets(file_path)
                if datasets:
                    dataset_name = datasets[0]['path']
                    print(f"✓ Using first matching dataset: {dataset_name}")
                else:
                    raise ValueError("No PECsw-related datasets found")
        
        if dataset_name not in f:
            raise ValueError(f"Dataset '{dataset_name}' not found")
        
        dataset = f[dataset_name]
        data = dataset[:]
        attrs = dict(dataset.attrs) if dataset.attrs else {}
        
        print(f"\n📊 Extracted Dataset Information:")
        print(f"   Path: {dataset_name}")
        print(f"   Shape: {data.shape}")
        print(f"   Dtype: {data.dtype}")
        print(f"   Min: {np.nanmin(data):.6e}, Max: {np.nanmax(data):.6e}")
        
        return data, attrs, dataset_name

try:
    pecsw_data, attributes, dataset_used = extract_pecsw_data(hdf5_path)
    print("\n✓ Data extraction successful!")
except Exception as e:
    print(f"\n✗ Error: {e}")
    pecsw_data = None

## 6. Extract Reach and Time Information

In [ ]:
def get_dimension_info(file_path):
    """Extract reach IDs and time information."""
    reach_ids = None
    time_info = None
    
    with h5py.File(file_path, 'r') as f:
        reach_paths = ['reaches', 'Reaches', 'reach_ids', 'ReachIds', 'CascadeToxswa/reaches']
        for path in reach_paths:
            if path in f:
                reach_ids = f[path][:]
                print(f"✓ Found reach IDs (n={len(reach_ids)})")
                break
        
        time_paths = ['time', 'Time', 'dates', 'Dates', 'CascadeToxswa/time']
        for path in time_paths:
            if path in f:
                time_info = f[path][:]
                print(f"✓ Found time info (n={len(time_info)})")
                break
    
    return reach_ids, time_info

reach_ids, time_info = get_dimension_info(hdf5_path)

## 7. Convert to DataFrame

In [ ]:
def create_pecsw_dataframe(pecsw_data, reach_ids=None, time_info=None, start_date='2000-01-01'):
    """Convert PECsw data to pandas DataFrame."""
    if pecsw_data is None:
        raise ValueError("No PECsw data provided")
    
    print(f"\n📊 Creating DataFrame from shape: {pecsw_data.shape}")
    
    if pecsw_data.ndim == 2:
        if pecsw_data.shape[1] > pecsw_data.shape[0] * 2:
            pecsw_data = pecsw_data.T
        
        n_times, n_reaches = pecsw_data.shape
        
        if reach_ids is not None and len(reach_ids) == n_reaches:
            columns = [f"Reach_{rid}" for rid in reach_ids]
        else:
            columns = [f"Reach_{i+1}" for i in range(n_reaches)]
        
        if time_info is not None and len(time_info) == n_times:
            try:
                index = pd.to_datetime(time_info)
            except:
                index = time_info
        else:
            index = pd.date_range(start=start_date, periods=n_times, freq='D')
        
        df = pd.DataFrame(pecsw_data, index=index, columns=columns)
        df.index.name = 'Time'
    
    elif pecsw_data.ndim == 3:
        print(f"  3D data - taking surface layer (index 0)")
        pecsw_2d = pecsw_data[:, :, 0]
        return create_pecsw_dataframe(pecsw_2d, reach_ids, time_info, start_date)
    
    else:
        df = pd.DataFrame({'PECsw': pecsw_data})
    
    print(f"✓ DataFrame created: {df.shape[0]} rows × {df.shape[1]} columns\n")
    return df

if pecsw_data is not None:
    df_pecsw = create_pecsw_dataframe(pecsw_data, reach_ids, time_info)
    print("="*80)
    print("DATAFRAME PREVIEW")
    print("="*80)
    display(df_pecsw.head(10))
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    display(df_pecsw.describe())

## 8. Export to Excel

In [ ]:
def export_to_excel(df, output_path):
    """Export DataFrame to Excel with multiple sheets."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📝 Exporting data to Excel...")
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        print("  - Writing 'PECsw' sheet...")
        df.to_excel(writer, sheet_name='PECsw')
        
        print("  - Writing 'Statistics' sheet...")
        stats = df.describe()
        stats.to_excel(writer, sheet_name='Statistics')
        
        print("  - Writing 'Max_Values' sheet...")
        max_values = pd.DataFrame({
            'Max_PECsw': df.max(),
            'Time_of_Max': df.idxmax(),
            'Mean_PECsw': df.mean(),
            'Std_PECsw': df.std()
        })
        max_values.to_excel(writer, sheet_name='Max_Values')
        
        print("  - Writing 'Time_Statistics' sheet...")
        time_stats = pd.DataFrame({
            'Max_Across_Reaches': df.max(axis=1),
            'Mean_Across_Reaches': df.mean(axis=1),
            'Min_Across_Reaches': df.min(axis=1)
        })
        time_stats.to_excel(writer, sheet_name='Time_Statistics')
    
    print(f"\n✓ Export successful!")
    print(f"  File: {output_path}")
    print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")
    return output_path

if pecsw_data is not None and 'df_pecsw' in locals():
    export_to_excel(df_pecsw, output_path)
else:
    print("⚠ No data available to export")

## 9. Visualizations (Optional)

In [ ]:
if pecsw_data is not None and 'df_pecsw' in locals():
    print("📊 Creating visualizations...\n")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Time series for first few reaches
    ax1 = axes[0, 0]
    n_reaches_to_plot = min(5, len(df_pecsw.columns))
    df_pecsw.iloc[:, :n_reaches_to_plot].plot(ax=ax1, linewidth=1.5)
    ax1.set_xlabel('Time', fontsize=10)
    ax1.set_ylabel('PECsw (µg/L)', fontsize=10)
    ax1.set_title(f'PECsw Time Series (First {n_reaches_to_plot} Reaches)', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Max PECsw per reach
    ax2 = axes[0, 1]
    max_per_reach = df_pecsw.max()
    max_per_reach.plot(kind='bar', ax=ax2, color='steelblue')
    ax2.set_xlabel('Reach', fontsize=10)
    ax2.set_ylabel('Max PECsw (µg/L)', fontsize=10)
    ax2.set_title('Maximum PECsw per Reach', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45, labelsize=8)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Spatial distribution (mean)
    ax3 = axes[1, 0]
    mean_per_reach = df_pecsw.mean()
    ax3.plot(range(len(mean_per_reach)), mean_per_reach, 'o-', linewidth=2, markersize=6, color='darkgreen')
    ax3.fill_between(range(len(mean_per_reach)), mean_per_reach, alpha=0.3, color='darkgreen')
    ax3.set_xlabel('Reach Number', fontsize=10)
    ax3.set_ylabel('Mean PECsw (µg/L)', fontsize=10)
    ax3.set_title('Mean PECsw Spatial Distribution', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Temporal summary
    ax4 = axes[1, 1]
    max_across_reaches = df_pecsw.max(axis=1)
    mean_across_reaches = df_pecsw.mean(axis=1)
    ax4.plot(df_pecsw.index, max_across_reaches, label='Max', linewidth=2, color='red', alpha=0.7)
    ax4.plot(df_pecsw.index, mean_across_reaches, label='Mean', linewidth=2, color='blue', alpha=0.7)
    ax4.fill_between(df_pecsw.index, mean_across_reaches, alpha=0.2, color='blue')
    ax4.set_xlabel('Time', fontsize=10)
    ax4.set_ylabel('PECsw (µg/L)', fontsize=10)
    ax4.set_title('Temporal Summary (Max and Mean Across All Reaches)', fontsize=12, fontweight='bold')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = output_path.parent / 'PECsw_plots.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"✓ Plots saved to: {fig_path}")
    
    plt.show()
else:
    print("⚠ No data available for visualization")

## 10. Summary Report

In [ ]:
if pecsw_data is not None and 'df_pecsw' in locals():
    print("\n" + "="*80)
    print("EXTRACTION SUMMARY")
    print("="*80)
    print(f"\nInput File: {hdf5_path.name}")
    print(f"Dataset Used: {dataset_used}")
    print(f"\nData Dimensions:")
    print(f"  - Number of time steps: {df_pecsw.shape[0]}")
    print(f"  - Number of reaches: {df_pecsw.shape[1]}")
    print(f"  - Total data points: {df_pecsw.size:,}")
    print(f"\nPECsw Statistics (µg/L):")
    print(f"  - Global maximum: {df_pecsw.max().max():.6e}")
    print(f"  - Global minimum: {df_pecsw.min().min():.6e}")
    print(f"  - Global mean: {df_pecsw.mean().mean():.6e}")
    print(f"  - Global std dev: {df_pecsw.std().mean():.6e}")
    print(f"\nReach with highest exposure:")
    max_reach = df_pecsw.max().idxmax()
    print(f"  - {max_reach}: {df_pecsw[max_reach].max():.6e} µg/L")
    print(f"\nTime of highest exposure across all reaches:")
    max_time = df_pecsw.max(axis=1).idxmax()
    print(f"  - {max_time}: {df_pecsw.max(axis=1).max():.6e} µg/L")
    print(f"\nOutput File: {output_path}")
    if output_path.exists():
        print(f"File Size: {output_path.stat().st_size / 1024:.1f} KB")
    print("\n" + "="*80)
    print("✓ EXTRACTION COMPLETE")
    print("="*80)
else:
    print("\n⚠ Data extraction was not successful")